# 从零理解 WordPiece：BERT 管线与 MaxMatch

## 学习目标

完成本 notebook 后，你应该能够：

1. 区分 normalization、BasicTokenizer、WordPiece 和特殊 token 后处理；
2. 从零实现经典 longest-match-first，并解释 `##` 的边界语义；
3. 说明一个字符缺失为何可能让整个 basic token 变成 `[UNK]`；
4. 使用 Trie 优化查找，并分析复杂度、Unicode、offset 与工程兼容问题；
5. 严谨比较 WordPiece、BPE 和 Unigram，而不混淆训练与推理。


## 1. 完整管线与训练思想

经典 BERT 管线可写成：原文 → 清理/大小写/重音处理 → 标点与 CJK 预分词 → 每个 basic token 做 WordPiece MaxMatch → 添加 `[CLS]`、`[SEP]` 等。

WordPiece 的历史思想是选择能改善训练数据概率或语言模型似然的子词。许多教学实现使用类似下式的关联评分：

\[
\operatorname{score}(a,b)=\frac{f(ab)}{f(a)f(b)}.
\]

这个公式有助于理解“不是单纯最高频合并”，但不同训练器的近似、剪枝和 tie-break 可能不同。推理阶段则通常不再使用这些分数，而只依赖最终词表做最长匹配。


In [ ]:
import unicodedata  # 导入本单元所需的依赖。

def is_cjk(char):  # 定义本节可复用的核心函数。
    code = ord(char)  # 计算并保存当前步骤的中间状态。
    return 0x4E00 <= code <= 0x9FFF  # 返回当前分支计算出的结果。

def normalize_token(token, lowercase=True, strip_accents=True):  # 定义本节可复用的核心函数。
    if lowercase:  # 按当前条件选择后续控制路径。
        token = token.lower()  # 计算并保存当前步骤的中间状态。
    if strip_accents:  # 按当前条件选择后续控制路径。
        token = "".join(  # 计算并保存当前步骤的中间状态。
            char for char in unicodedata.normalize("NFD", token)  # 执行当前语句以推进本节示例。
            if unicodedata.category(char) != "Mn"  # 按当前条件选择后续控制路径。
        )  # 执行当前语句以推进本节示例。
    return token  # 返回当前分支计算出的结果。

def basic_tokenize(text, lowercase=True, strip_accents=True):  # 定义本节可复用的核心函数。
    tokens, current = [], []  # 计算并保存当前步骤的中间状态。
    def flush():  # 定义本节可复用的核心函数。
        if current:  # 按当前条件选择后续控制路径。
            token = normalize_token("".join(current), lowercase, strip_accents)  # 计算并保存当前步骤的中间状态。
            if token:  # 按当前条件选择后续控制路径。
                tokens.append(token)  # 执行当前语句以推进本节示例。
            current.clear()  # 执行当前语句以推进本节示例。
    for char in text:  # 遍历输入元素以累积或检查结果。
        category = unicodedata.category(char)  # 计算并保存当前步骤的中间状态。
        if char.isspace():  # 按当前条件选择后续控制路径。
            flush()  # 执行当前语句以推进本节示例。
        elif is_cjk(char) or category.startswith("P"):  # 按当前条件选择后续控制路径。
            flush()  # 执行当前语句以推进本节示例。
            tokens.append(char)  # 执行当前语句以推进本节示例。
        elif not category.startswith("C"):  # 按当前条件选择后续控制路径。
            current.append(char)  # 执行当前语句以推进本节示例。
    flush()  # 执行当前语句以推进本节示例。
    return tokens  # 返回当前分支计算出的结果。

sample = "Hello,世界! Résumé playing"  # 计算并保存当前步骤的中间状态。
print(basic_tokenize(sample))  # 执行当前语句以推进本节示例。


## 2. `##` 与 longest-match-first

`play` 只能匹配一个 basic token 的开头，`##ing` 只能匹配词内续接位置。算法从左到右，在当前位置从最长子串开始缩短；词首直接查表，后续加 `##` 后查表。命中后前进到终点。若任何位置连单字符候选都不存在，经典实现把整个词返回为 `[UNK]`，而不是保留已经匹配的前缀。

最长匹配是局部规则：它不比较完整路径概率，也不保证 token 数全局最少。


In [ ]:
def wordpiece_encode(word, vocab, unk_token="[UNK]", max_chars=100):  # 定义本节可复用的核心函数。
    if len(word) > max_chars:  # 按当前条件选择后续控制路径。
        return [unk_token]  # 返回当前分支计算出的结果。
    pieces = []  # 计算并保存当前步骤的中间状态。
    start = 0  # 计算并保存当前步骤的中间状态。
    while start < len(word):  # 在终止条件满足前持续推进状态。
        end = len(word)  # 计算并保存当前步骤的中间状态。
        matched = None  # 计算并保存当前步骤的中间状态。
        while end > start:  # 在终止条件满足前持续推进状态。
            substring = word[start:end]  # 计算并保存当前步骤的中间状态。
            candidate = substring if start == 0 else "##" + substring  # 计算并保存当前步骤的中间状态。
            if candidate in vocab:  # 按当前条件选择后续控制路径。
                matched = candidate  # 计算并保存当前步骤的中间状态。
                break  # 调整当前循环或占位控制流。
            end -= 1  # 计算并保存当前步骤的中间状态。
        if matched is None:  # 按当前条件选择后续控制路径。
            return [unk_token]  # 返回当前分支计算出的结果。
        pieces.append(matched)  # 执行当前语句以推进本节示例。
        start = end  # 计算并保存当前步骤的中间状态。
    return pieces  # 返回当前分支计算出的结果。

vocab = {  # 计算并保存当前步骤的中间状态。
    "[UNK]", "hello", ",", "世", "界", "!", "resume",  # 执行当前语句以推进本节示例。
    "play", "##ing", "player", "un", "##aff", "##able",  # 执行当前语句以推进本节示例。
    "a", "ab", "##bc", "##c"  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

for word in ["playing", "player", "unaffable", "abc", "play🙂"]:  # 遍历输入元素以累积或检查结果。
    print(f"{word!r:12} -> {wordpiece_encode(word, vocab)}")  # 执行当前语句以推进本节示例。


## 3. 逐步反例：最长不等于全局最优

对 `abc`，词表同时包含 `ab`、`a`、`##bc` 和 `##c`。MaxMatch 首步会选择更长的 `ab`，随后选择 `##c`，结果是 `[ab, ##c]`；它不会回头比较 `[a, ##bc]`。如果需要根据完整路径概率选最优，应使用 Unigram 的 lattice + Viterbi，而不是修改 WordPiece 定义。

另外，WordPiece 只在一个 BasicTokenizer 片段内搜索，绝不会跨越上游已经建立的空格、标点或 CJK 边界。


In [ ]:
def tokenize(text, vocab):  # 定义本节可复用的核心函数。
    basic_tokens = basic_tokenize(text)  # 计算并保存当前步骤的中间状态。
    pieces = []  # 计算并保存当前步骤的中间状态。
    for token in basic_tokens:  # 遍历输入元素以累积或检查结果。
        pieces.extend(wordpiece_encode(token, vocab))  # 执行当前语句以推进本节示例。
    return basic_tokens, pieces  # 返回当前分支计算出的结果。

for text in ["Hello,世界!", "Résumé playing", "abc"]:  # 遍历输入元素以累积或检查结果。
    basic, pieces = tokenize(text, vocab)  # 计算并保存当前步骤的中间状态。
    print("原文:", text)  # 执行当前语句以推进本节示例。
    print("BasicTokenizer:", basic)  # 执行当前语句以推进本节示例。
    print("WordPiece:", pieces, "\n")  # 执行当前语句以推进本节示例。


## 4. 用 Trie 避免反复构造子串

朴素实现在每个起点反复缩短终点，最坏会产生 $O(n^2)$ 级尝试与 substring 分配。Trie 共享词表前缀：从当前位置向前走，记录最后一个终止节点，路径失败时回退到这个“最远合法终点”。词首与续接使用不同根，保持 `##` 状态。


In [ ]:
END = "<END>"  # 计算并保存当前步骤的中间状态。

def build_trie(tokens):  # 定义本节可复用的核心函数。
    root = {}  # 计算并保存当前步骤的中间状态。
    for token in tokens:  # 遍历输入元素以累积或检查结果。
        node = root  # 计算并保存当前步骤的中间状态。
        for char in token:  # 遍历输入元素以累积或检查结果。
            node = node.setdefault(char, {})  # 计算并保存当前步骤的中间状态。
        node[END] = token  # 计算并保存当前步骤的中间状态。
    return root  # 返回当前分支计算出的结果。

start_trie = build_trie(token for token in vocab if not token.startswith("##"))  # 计算并保存当前步骤的中间状态。
cont_trie = build_trie(token[2:] for token in vocab if token.startswith("##"))  # 计算并保存当前步骤的中间状态。

def longest_from(word, start, trie):  # 定义本节可复用的核心函数。
    node = trie  # 计算并保存当前步骤的中间状态。
    best_end = None  # 计算并保存当前步骤的中间状态。
    for index in range(start, len(word)):  # 遍历输入元素以累积或检查结果。
        char = word[index]  # 计算并保存当前步骤的中间状态。
        if char not in node:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        node = node[char]  # 计算并保存当前步骤的中间状态。
        if END in node:  # 按当前条件选择后续控制路径。
            best_end = index + 1  # 计算并保存当前步骤的中间状态。
    return best_end  # 返回当前分支计算出的结果。

def wordpiece_encode_trie(word, unk_token="[UNK]", max_chars=100):  # 定义本节可复用的核心函数。
    if len(word) > max_chars:  # 按当前条件选择后续控制路径。
        return [unk_token]  # 返回当前分支计算出的结果。
    pieces, start = [], 0  # 计算并保存当前步骤的中间状态。
    while start < len(word):  # 在终止条件满足前持续推进状态。
        trie = start_trie if start == 0 else cont_trie  # 计算并保存当前步骤的中间状态。
        end = longest_from(word, start, trie)  # 计算并保存当前步骤的中间状态。
        if end is None:  # 按当前条件选择后续控制路径。
            return [unk_token]  # 返回当前分支计算出的结果。
        surface = word[start:end]  # 计算并保存当前步骤的中间状态。
        pieces.append(surface if start == 0 else "##" + surface)  # 计算并保存当前步骤的中间状态。
        start = end  # 计算并保存当前步骤的中间状态。
    return pieces  # 返回当前分支计算出的结果。

for word in ["playing", "unaffable", "abc", "play🙂"]:  # 遍历输入元素以累积或检查结果。
    expected = wordpiece_encode(word, vocab)  # 计算并保存当前步骤的中间状态。
    actual = wordpiece_encode_trie(word)  # 计算并保存当前步骤的中间状态。
    print(word, actual)  # 执行当前语句以推进本节示例。
    assert actual == expected  # 用受控断言验证关键不变量。


## 5. 边界、误区与相邻算法对比

- **`##` 不是原文字符**，offset 不能把两个井号计入跨度；
- **整体 `[UNK]`**：中途失败时经典行为会丢掉整个 basic token 的内部信息；
- **Normalization 是 ABI**：cased/uncased、去重音和 CJK 规则必须与 checkpoint 一致；
- **Trie 不能一见终止节点就输出**，必须继续寻找最远终止点；
- **特殊 token 不由 MaxMatch 产生**，通常由保护规则或 post-processor 插入；
- **WordPiece 不等于 WWM**：Whole Word Masking 是预训练 mask 策略。

| 方法 | 典型训练 | 典型推理 | 未知输入 |
|---|---|---|---|
| WordPiece | 似然收益或实现近似扩词表 | 局部最长匹配 | 经典实现可能整词 `[UNK]` |
| BPE | 合并最高频相邻 pair | 按 merge rank | byte 底座可无 OOV |
| Unigram | EM 学概率并剪枝 | 全局 Viterbi | required chars/byte fallback 决定覆盖 |


In [ ]:
# 一组可以放进 CI 的最小语义测试
assert wordpiece_encode("abc", vocab) == ["ab", "##c"]  # 用受控断言验证关键不变量。
assert wordpiece_encode("play🙂", vocab) == ["[UNK]"]  # 用受控断言验证关键不变量。
assert wordpiece_encode("a" * 101, vocab, max_chars=100) == ["[UNK]"]  # 用受控断言验证关键不变量。
assert wordpiece_encode_trie("playing") == ["play", "##ing"]  # 用受控断言验证关键不变量。
print("全部边界测试通过。")  # 执行当前语句以推进本节示例。


## 练习与面试总结

1. 为 BasicTokenizer 增加可配置 `never_split`，确保 `[MASK]` 不被 lowercase。
2. 为 Trie 编码器返回相对字符 offsets，再考虑 normalization 到原文的映射。
3. 生成随机小词表，对朴素版和 Trie 版做 property-based 对拍。
4. 统计一批领域文本的 fertility、`[UNK]` 率和 P99 单词长度。

**一分钟回答**：经典 WordPiece 先由 BasicTokenizer 建立词边界，再对每个词从左到右做最长匹配；词内续接项带 `##`。任一位置没有候选时通常整词变 `[UNK]`。它与 BPE 的 merge-rank 编码、Unigram 的概率 Viterbi 都不同。生产上必须把 vocab、大小写/重音/CJK 配置、特殊 token、长度上限和 offset 单位与 BERT checkpoint 一起冻结。
